[![Mở trong Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/trungnguyenvn/mlsysbook-vi-notebooks/blob/main/vol1-ch04-data-engineering/01-du-lieu-co-khoi-luong.ipynb)

# Dữ liệu có khối lượng: thời gian, năng lượng và tiền của việc di chuyển

Chương 4 coi dữ liệu như một chất vật lý. Nó có khối lượng, nên chuyển nó đi mất thời gian và
tiền. Chuyển nó qua từng tầng của hệ thống tốn năng lượng. Rót nó vào một bộ tăng tốc
(accelerator) cần một lưu lượng mà không phải tầng lưu trữ nào cũng cấp nổi. Chương đặt mỗi ý đó
thành một phép tính trên khăn giấy. Sổ tay này làm lại từng phép, rồi đẩy mỗi phép xa hơn một
bước: quét một tham số, vẽ đường cong, xem chỗ nào kết luận đổi chiều.

**Bạn sẽ làm:**
1. tính trọng lực dữ liệu (data gravity): chuyển 1 PB mất bao lâu, tốn bao nhiêu, và khi nào nên
   chuyển compute tới dữ liệu;
2. đọc bảng năng lượng của chương cạnh số phép tính mà mô hình làm trên mỗi giá trị;
3. tính thuế tiếp liệu (feeding tax) và ngân sách băng thông (bandwidth) lưu trữ cho một bộ tăng
   tốc huấn luyện ResNet-50;
4. đặt tập dữ liệu phát hiện từ khoá (keyword spotting) lên năm tầng lưu trữ và lên bậc thang độ
   trễ (latency);
5. tính giá của nạp theo luồng so với nạp theo lô, và của ETL so với ELT, rồi tìm điểm hoà vốn.

In [ ]:
#@title Chuẩn bị: thư viện, hạt giống và hai hàm đối chiếu với sách { display-mode: "form" }
# Chạy ô này trước. Nó không cài thêm gì: mọi thư viện dùng ở đây đều có sẵn trên Colab.
import math, os, random, re, sys, time

import matplotlib.pyplot as plt
import numpy as np
import torch

# Số luồng CPU. 0 để thư viện tự chọn (trên Colab là 2).
NB_THREADS = int(os.environ.get("NB_THREADS", "0"))
if NB_THREADS:
    torch.set_num_threads(NB_THREADS)
    try:
        torch.set_num_interop_threads(NB_THREADS)
    except RuntimeError:  # chỉ đặt được một lần mỗi phiên
        pass
    from threadpoolctl import threadpool_limits
    threadpool_limits(NB_THREADS)

SEED = 42


def dat_hat_giong(seed=SEED):
    """Cùng hạt giống thì cùng kết quả, mỗi lần chạy lại."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


dat_hat_giong()
DATA = os.environ.get("NB_DATA", "data")  # nơi tải dữ liệu về
NB_STRICT = os.environ.get("NB_STRICT") == "1"
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 110,
                     "axes.grid": True, "grid.alpha": 0.3})

# Đọc một con số theo cách bảng số liệu của bài học viết nó (dấu chấm hay dấu phẩy).
import math
import re

_SUP = str.maketrans("⁰¹²³⁴⁵⁶⁷⁸⁹⁻⁺", "0123456789-+")
_MULT = {"tỷ": 1e9, "tỉ": 1e9, "triệu": 1e6, "nghìn": 1e3, "ngàn": 1e3,
         "K": 1e3, "M": 1e6, "B": 1e9}
_NUM = re.compile(
    r"(?<![\w.,])(-?)(\d+(?:[.,]\d+)*)"                      # sign, digits with separators
    r"(?:\s*[×x·]\s*10\s*(?:\^\{?\s*([-+]?\d+)\s*\}?|([⁻⁺]?[⁰¹²³⁴⁵⁶⁷⁸⁹]+))|[eE]([-+]?\d+)"
    r"|\^\{?\s*([-+]?\d+)\s*\}?|([⁻⁺]?[⁰¹²³⁴⁵⁶⁷⁸⁹]+))?"                  # or a bare power, 10^{-5}
    r"(\s*%|\s*(?:tỷ|tỉ|triệu|nghìn|ngàn)(?!\w)|\s?[KMB](?![A-Za-z]))?")


def norm(text):
    """The outline's markup and typography reduced to plain text, as both sides compare it."""
    text = text.replace("{,}", ",").replace("**", "").replace("`", "")
    text = re.sub(r"[   ]", " ", text)
    text = re.sub(r"−(?=\d)", "-", text)
    text = re.sub(r"[~≈]", "", text)
    return re.sub(r"\s+", " ", text).strip()


def _readings(digits):
    """(value, decimals) for each way the separators in `digits` can be read."""
    out = []
    groups_ok = lambda parts: all(len(p) == 3 for p in parts[1:])
    for thou, dec in ((",", "."), (".", ",")):
        if thou in digits and dec in digits:
            head, _, tail = digits.rpartition(dec)
            if dec in head or not tail.isdigit() or not groups_ok(head.split(thou)):
                continue
            out.append((float(head.replace(thou, "") + "." + tail), len(tail)))
    for sep in (",", "."):
        other = "." if sep == "," else ","
        if sep in digits and other not in digits:
            parts = digits.split(sep)
            if groups_ok(parts):
                out.append((float("".join(parts)), 0))                    # thousands
            if len(parts) == 2:
                out.append((float(parts[0] + "." + parts[1]), len(parts[1])))  # decimal
    if "." not in digits and "," not in digits:
        out.append((float(digits), 0))
    return out


def candidates(text):
    """Every (value, half_unit) the numbers in `text` can mean.

    half_unit is half of the last printed digit, scaled like the value, so `437,5 KB`
    accepts 437.544. Each number is offered as written and, when a multiplier or `%`
    follows it, scaled by that too, so `8,5 tỉ` cites 8.5 or 8.5e9 and `20 %` cites 20 or 0.2.
    """
    out = []
    for m in _NUM.finditer(norm(text)):
        sign, digits, exp_caret, exp_sup, exp_e, pow_caret, pow_sup, suffix = m.groups()
        exp = exp_caret or (exp_sup.translate(_SUP) if exp_sup else None) or exp_e
        power = pow_caret or (pow_sup.translate(_SUP) if pow_sup else None)
        scale10 = 10.0 ** int(exp) if exp else 1.0
        suffix = (suffix or "").strip()
        extra = 0.01 if suffix == "%" else _MULT.get(suffix)
        for value, decimals in _readings(digits):
            if power:
                # 10^{-5}, 2³²: an exact value, so it is matched to 1 part in 10⁹; a looser
                # comparison needs an explicit tol.
                v = value ** int(power)
                half = abs(v) * 1e-9
            else:
                v = value * scale10
                half = 0.5 * 10.0 ** -decimals * scale10
            v = -v if sign else v
            out.append((v, half))
            if extra:
                out.append((v * extra, half * extra))
    return out


def match(value, text):
    """The (value, half_unit) reading of `text` that `value` equals, or None."""
    for v, half in candidates(text):
        if math.isclose(value, v, rel_tol=1e-9, abs_tol=1e-300):
            return v, half
    return None


def well_anchored(text):
    """False for a citation too bare to identify one cell, like "1" or "2"."""
    t = norm(text)
    rest = re.sub(r"[\d.,\s\-+]", "", t)
    if rest:
        return True
    sig = re.sub(r"[^\d]", "", t).lstrip("0")
    return len(sig) >= 2


def _so(x):
    """Một con số để in: số nguyên có dấu phẩy ngăn nghìn, số thực giữ 6 chữ số có nghĩa."""
    if isinstance(x, (int, np.integer)) or (isinstance(x, float) and x.is_integer() and abs(x) < 1e15):
        return f"{int(x):,}"
    return f"{x:,.6g}"


def sach(gia_tri, nguon=None, trich=None, tol=None):
    """Một con số của sách, kèm đúng chữ của nó: trong bảng số liệu của bài học (nguon)
    hoặc nguyên văn trong chương (trich)."""
    chu = nguon if nguon is not None else trich
    if match(gia_tri, chu) is None:
        raise ValueError(f"{gia_tri!r} không phải con số ghi trong {chu!r}")
    return gia_tri


def theo_sach(ten, gia_tri, sach, nguon=None, trich=None, tol=None):
    """So một con số tính được với con số của sách. In ✓ khi khớp, ✗ kèm độ lệch khi không."""
    chu = nguon if nguon is not None else trich
    khop = match(sach, chu)
    if khop is None:
        raise ValueError(f"{sach!r} không phải con số ghi trong {chu!r}")
    sai_so = tol if tol is not None else khop[1]
    dung = abs(gia_tri - sach) <= sai_so
    print(f"{'✓' if dung else '✗'} {ten}: tính được {_so(gia_tri)} · sách: {chu}")
    if not dung:
        loi = f"lệch {_so(gia_tri - sach)}, quá sai số cho phép {_so(sai_so)}"
        if NB_STRICT:
            raise AssertionError(f"{ten}: {loi}")
        print(f"   {loi}. Nếu bạn vừa đổi tham số ở trên thì đây là điều được chờ đợi.")
    # None, not the bool: a cell ending in theo_sach(...) would otherwise echo True under the ✓.


def do_tren_may(ten, gia_tri, don_vi=""):
    """Một con số ĐO trên máy đang chạy: mỗi máy một khác, không phải con số của sách."""
    print(f"⏱ {ten}: {_so(gia_tri)} {don_vi} (đo trên máy này)".replace("  ", " "))


print(f"Python {sys.version.split()[0]} · PyTorch {torch.__version__} · NumPy {np.__version__}"
      f" · {torch.get_num_threads()} luồng CPU")

In [ ]:
import pandas as pd

MAU = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]  # thứ tự màu cố định
GB, TB, PB = 1e9, 1e12, 1e15                         # byte, theo quy ước thập phân của chương
NGAY = 86_400                                         # giây


def doc_thoi_gian(giay):
    """Một khoảng thời gian, in theo đơn vị dễ đọc nhất."""
    for don_vi, s in (("năm", 365 * NGAY), ("ngày", NGAY), ("giờ", 3600), ("phút", 60), ("giây", 1)):
        if giay >= s * (1 - 1e-9):   # 0.5 ns × 2 × 10⁹ ra 0.9999999 chứ không phải 1
            return f"{giay / s:,.1f} {don_vi}"
    return f"{giay * 1000:,.1f} ms"


print("sẵn sàng:", doc_thoi_gian(90_000), "·", doc_thoi_gian(0.25))

## 1. Trọng lực dữ liệu: chuyển dữ liệu hay chuyển compute

Chương định nghĩa trọng lực dữ liệu là chi phí của việc di chuyển, một hàm của khối lượng dữ liệu
$D_\text{vol}$ và băng thông mạng $\text{BW}$:

$$T = \frac{D_\text{vol}}{\text{BW}}.$$

Một chi tiết hay làm sai: băng thông mạng tính bằng **bit** mỗi giây, còn dung lượng tính bằng
**byte**, nên phải chia cho 8.

**Dự đoán:** chuyển 1 PB qua một đường 10 Gb/s mất bao lâu? Một giờ, một ngày, hay hơn một tuần?

In [ ]:
KHOI = sach(1, nguon="1 PB · 10 Gb/s · 9,3 ngày") * PB
DUONG_10 = sach(10, nguon="1 PB · 10 Gb/s · 9,3 ngày") * 1e9 / 8   # bit/s → byte/s
theo_sach("1 PB qua 10 Gb/s (ngày)", KHOI / DUONG_10 / NGAY, sach=9.3, nguon="1 PB · 10 Gb/s · 9,3 ngày")

Napkin math 1.1 của chương đặt câu hỏi cụ thể hơn: tập dữ liệu 1 PB nằm ở miền đông nước Mỹ, cụm
tăng tốc đang rảnh nằm ở miền tây, và hai nơi nối bằng một đường riêng 100 Gb/s.

In [ ]:
BW_100 = sach(100, nguon="100 Gb/s ÷ 8 = 12,5 GB/s") * 1e9 / 8
theo_sach("100 Gb/s chia 8 (GB/s)", BW_100 / GB, sach=12.5, nguon="100 Gb/s ÷ 8 = 12,5 GB/s")
KHOI_GB = sach(1_000_000, nguon="12,5 GB/s · 1.000.000 GB")
giay_truyen = KHOI_GB * GB / BW_100
theo_sach("thời gian truyền (giây)", giay_truyen, sach=80_000, nguon="80.000 giây ≈ 22 giờ", tol=1)
theo_sach("thời gian truyền (giờ)", giay_truyen / 3600, sach=22, nguon="80.000 giây ≈ 22 giờ", tol=0.5)

GIA_RA = sach(0.02, nguon="0,02 đô la/GB · 20.000 đô la")   # phí truyền ra, bảng giá 2024
phi_ra = KHOI_GB * GIA_RA
theo_sach("phí truyền ra ($)", phi_ra, sach=20_000, nguon="0,02 đô la/GB · 20.000 đô la")
GIO_TPU = sach(5000, nguon="20.000 đô la · ~5.000 giờ-TPUv4", tol=500)
print(f"suy ra: chương quy {phi_ra:,.0f} $ ra khoảng {GIO_TPU:,} giờ TPUv4, "
      f"tức khoảng {phi_ra / GIO_TPU:.0f} $ mỗi giờ (chương không ghi giá này)")

Từ hai phép chia đó ra hai lằn ranh: nếu huấn luyện ngắn hơn 22 giờ thì truyền dữ liệu lâu hơn cả
huấn luyện, và nếu huấn luyện rẻ hơn 20,000 $ thì băng thông đắt hơn compute. Với dữ liệu cỡ
petabyte, mã nguồn chuyển tới chỗ dữ liệu; với dữ liệu cỡ gigabyte, dữ liệu chuyển tới chỗ mã nguồn.

Hình dưới quét khối lượng từ 1 GB tới 1 PB trên ba đường truyền. Đường 1 Gb/s là mức sổ tay thêm
vào để so; hai điểm có nhãn là hai con số của chương.

In [ ]:
khoi_luong = np.logspace(0, 6, 61) * GB
duong = {"1 Gb/s": 1e9 / 8, "10 Gb/s": DUONG_10, "100 Gb/s": BW_100}

fig, ax = plt.subplots(figsize=(8, 4.4))
for (ten, bw), mau in zip(duong.items(), MAU):
    ax.loglog(khoi_luong / GB, khoi_luong / bw / 3600, color=mau, lw=2, label=ten)
for nhan, gio in (("1 giờ", 1), ("1 ngày", 24), ("1 tuần", 168)):
    ax.axhline(gio, color="0.6", lw=0.8, ls=":")
    ax.text(1.2, gio * 1.12, nhan, fontsize=8, color="0.35")
for bw, chu in ((DUONG_10, f"1 PB, 10 Gb/s:\n{KHOI / DUONG_10 / NGAY:.1f} ngày"),
                (BW_100, f"1 PB, 100 Gb/s:\n{KHOI / BW_100 / 3600:.0f} giờ")):
    ax.plot(KHOI / GB, KHOI / bw / 3600, "o", color="0.15", ms=7, mec="white", mew=1.2)
    ax.annotate(chu, (KHOI / GB, KHOI / bw / 3600), xytext=(9, 0), textcoords="offset points",
                fontsize=8, va="center")
ax.set_xlim(0.5, 5e7)
ax.set_xlabel("khối lượng dữ liệu (GB, thang log)")
ax.set_ylabel("thời gian truyền (giờ, thang log)")
ax.set_title("Trọng lực dữ liệu: T = khối lượng / băng thông")
ax.legend(loc="lower right", fontsize=8, frameon=False)
plt.tight_layout()
plt.show()

Cùng một đường thẳng trên thang log, chỉ dịch theo băng thông: nhân băng thông lên 10 thì thời gian
chia 10, nhưng khối lượng tăng nhanh hơn nhiều. Chương còn nêu một cạm bẫy: lập kế hoạch di chuyển
cỡ petabyte như thể chỉ là một phép chia này. Thời gian trên dây là phần dễ tính nhất; dựng lại quy
trình, kiểm định lại chất lượng, di chuyển giản đồ và cập nhật phả hệ thường tốn thời gian kỹ sư hơn
hẳn.

## 2. Bất biến năng lượng di chuyển

Bất biến năng lượng di chuyển (energy-movement invariant) của chương nói rằng di chuyển một giá trị 32
bit tốn năng lượng gấp nhiều lần một phép nhân trên nó, và hệ số đó tăng khi dữ liệu rời xa con chip.
Bảng 1 cho năng lượng của một sự kiện ở mỗi tầng, cùng một độ rộng 32 bit, quy về một phép nhân dấu
phẩy động 32 bit:

In [ ]:
E_NHAN = sach(3.7, nguon="3,7 pJ/FLOP · 640 pJ · 4.000 pJ · 40.000 pJ")
E_TANG = {"truy cập DRAM": sach(640, nguon="3,7 pJ/FLOP · 640 pJ · 4.000 pJ · 40.000 pJ"),
          "truy cập SSD cục bộ": sach(4000, nguon="3,7 pJ/FLOP · 640 pJ · 4.000 pJ · 40.000 pJ"),
          "truyền qua mạng": sach(40_000, nguon="3,7 pJ/FLOP · 640 pJ · 4.000 pJ · 40.000 pJ")}
theo_sach("DRAM so với phép nhân", E_TANG["truy cập DRAM"] / E_NHAN, sach=173, nguon="173× · 1.081,1× · 10.810,8×")
theo_sach("SSD so với phép nhân", E_TANG["truy cập SSD cục bộ"] / E_NHAN, sach=1081.1,
          nguon="173× · 1.081,1× · 10.810,8×")
theo_sach("mạng so với phép nhân", E_TANG["truyền qua mạng"] / E_NHAN, sach=10810.8,
          nguon="173× · 1.081,1× · 10.810,8×")

Chương dặn kỹ: bảng này **tự nó không chứng minh** số hạng nào chi phối cả tải công việc
(workload), vì tổng năng lượng còn tuỳ số phép tính và số lần truyền. Vậy hãy đặt thêm số phép
tính vào. Nếu mô hình làm $F$ FLOP trên mỗi giá trị 32 bit nó đọc, thì phần năng lượng dành cho di
chuyển là

$$\frac{E_\text{di chuyển}}{E_\text{di chuyển} + F \cdot E_\text{FLOP}},$$

và hai phần bằng nhau đúng khi $F$ bằng các bội số vừa tính. Hai điểm để đối chiếu: ResNet-50 khi
suy luận, dùng hai con số của napkin math 1.9 (khoảng 8 GFLOP mỗi lượt xuôi, ảnh JPEG 150 KB), và
một mô hình tuyến tính trên dữ liệu bảng, mà sổ tay **giả định** làm 2 FLOP (một nhân, một cộng)
trên mỗi giá trị.

**Dự đoán:** khi đọc ảnh từ SSD, năng lượng của ResNet-50 nằm chủ yếu ở việc đọc hay ở việc tính?

In [ ]:
FLOP_XUOI = sach(8, nguon="312 TFLOP/s · ~8 GFLOP", tol=0.5) * 1e9
ANH_BYTE = sach(150, nguon="12.682 img/s · 150 KB") * 1e3
F_RESNET = FLOP_XUOI / (ANH_BYTE / 4)       # FLOP trên mỗi giá trị 32 bit của ảnh đã nén
F_TUYEN_TINH = 2                            # giả định của sổ tay
print(f"ResNet-50: {F_RESNET:,.0f} FLOP trên mỗi giá trị 32 bit đọc vào")
for ten, e in E_TANG.items():
    print(f"  {ten:>20}: hoà ở {e / E_NHAN:>9,.1f} FLOP/giá trị · "
          f"ResNet-50 tốn cho di chuyển {100 * e / (e + F_RESNET * E_NHAN):.3f} % · "
          f"mô hình tuyến tính {100 * e / (e + F_TUYEN_TINH * E_NHAN):.1f} %")

cac_F = np.logspace(0, 6, 300)
fig, ax = plt.subplots(figsize=(8, 4.2))
for (ten, e), mau in zip(E_TANG.items(), MAU):
    ax.semilogx(cac_F, 100 * e / (e + cac_F * E_NHAN), color=mau, lw=2, label=ten)
for F, chu in ((F_TUYEN_TINH, "mô hình tuyến tính\n(giả định 2 FLOP)"), (F_RESNET, "ResNet-50\nsuy luận")):
    ax.axvline(F, color="0.45", lw=1, ls="--")
    ax.text(F * 1.15, 55, chu, fontsize=8, color="0.25")
ax.set_ylim(0, 105)
ax.set_xlabel("FLOP mô hình làm trên mỗi giá trị 32 bit đọc vào (thang log)")
ax.set_ylabel("phần năng lượng cho di chuyển (%)")
ax.set_title("Cùng bảng năng lượng, kết luận đổi theo số phép tính")
ax.legend(loc="lower left", fontsize=8, frameon=False)
plt.tight_layout()
plt.show()

Với ResNet-50, phần đọc ảnh chỉ là một mẩu nhỏ của năng lượng, kể cả khi đọc qua mạng: mỗi byte
vào kéo theo rất nhiều phép tính. Với một mô hình đơn giản trên dữ liệu bảng, gần như toàn bộ năng
lượng nằm ở việc đưa dữ liệu tới, nên mỗi lần truyền tránh được là gần như toàn bộ phần tiết kiệm.
Đó là lý do chương nói tính cục bộ của dữ liệu (data locality) **có thể** là một đòn bẩy tối ưu
lớn, chứ không nói nó luôn là đòn bẩy lớn nhất.

## 3. Bài toán tiếp liệu và thuế tiếp liệu

Trọng lực dữ liệu là chi phí chuyển cả khối lượng. Bài toán tiếp liệu (feeding problem) là chi phí
giao khối lượng đó, từng mẻ một. Chương đặt một bộ tăng tốc chạy mô hình ảnh xử lý được 1,843 img/s,
trong khi quy trình lưu trữ chỉ đưa được 250 MB/s, và nói muốn rót đủ cho nó thường cần 1.1 GB/s.
Thuế tiếp liệu là phần thời gian bị mất vì chờ vào ra:

$$\text{thuế tiếp liệu} = 1 - \frac{\text{băng thông cấp được}}{\text{băng thông cần}}.$$

**Dự đoán:** với 250 MB/s so với 1.1 GB/s, bộ tăng tốc nhàn rỗi bao nhiêu phần trăm thời gian?

In [ ]:
R_ANH = sach(1843, nguon="1.843 img/s vs 250 MB/s · thuế tiếp liệu > 77,5 % · cần 1,1 GB/s")
CAP = sach(250, nguon="1.843 img/s vs 250 MB/s · thuế tiếp liệu > 77,5 % · cần 1,1 GB/s") * 1e6
CAN = sach(1.1, nguon="1.843 img/s vs 250 MB/s · thuế tiếp liệu > 77,5 % · cần 1,1 GB/s") * GB
THUE_SACH = sach(77.5, nguon="1.843 img/s vs 250 MB/s · thuế tiếp liệu > 77,5 % · cần 1,1 GB/s")

thue = 1 - CAP / CAN
print(f"thuế tiếp liệu tính từ 250 MB/s và 1.1 GB/s: {100 * thue:.1f} %  ·  chương: hơn {THUE_SACH} %")
can_cho_77_5 = CAP / (1 - THUE_SACH / 100)
print(f"để thuế đúng {THUE_SACH} %, nhu cầu phải là {can_cho_77_5 / GB:.3f} GB/s, "
      f"tức 1.1 GB/s đã được làm tròn")
byte_moi_anh = CAN / R_ANH
print(f"suy ra mỗi ảnh khoảng {byte_moi_anh / 1e3:,.0f} KB (chương không ghi con số này)")

Phép tính của chương không khép kín ở chữ số cuối: 250 MB/s chia cho đúng 1.1 GB/s cho thuế 77.3 %,
thấp hơn một chút so với "hơn 77.5 %". Hai con số khớp nếu nhu cầu thật là khoảng 1.11 GB/s, tức
1.1 là bản làm tròn. Kết luận không đổi: bộ tăng tốc dành phần lớn thời gian để chờ bit. Sổ tay in
cả hai và không khẳng định bên nào.

Chương tổng quát hoá điều này thành hai phương trình. Phương trình 5 nói thông lượng (throughput)
huấn luyện bị chặn bởi số nhỏ hơn giữa năng lực compute và tốc độ cấp dữ liệu; phương trình 6 nói
tốc độ cấp dữ liệu là băng thông lưu trữ, trừ phần mất cho phụ trội, chia cho số byte mỗi mẫu:

$$R_\text{train} \le \min(R_\text{compute}, R_\text{data}), \qquad
  R_\text{data} = \frac{B_\text{storage}\,(1 - \eta_\text{overhead})}{S_\text{sample}}.$$

In [ ]:
def toc_do_huan_luyen(R_compute, B_storage, S_sample, eta_overhead=0.0):
    """Phương trình 5 và 6: mẫu mỗi giây mà huấn luyện đạt được, và tầng nào chặn."""
    R_data = B_storage * (1 - eta_overhead) / S_sample
    return min(R_compute, R_data), ("dữ liệu" if R_data < R_compute else "compute")


cac_B = np.logspace(7, 10.5, 300)
fig, ax = plt.subplots(figsize=(8, 4.2))
for eta, mau, nhan in ((0.0, MAU[0], "không phụ trội"), (0.3, MAU[1], "giả định 30 % phụ trội")):
    muc = [100 * toc_do_huan_luyen(R_ANH, B, byte_moi_anh, eta)[0] / R_ANH for B in cac_B]
    ax.semilogx(cac_B / 1e6, muc, color=mau, lw=2, label=nhan)
ax.axvline(CAP / 1e6, color="0.4", lw=1, ls="--")
ax.text(CAP / 1e6 * 1.08, 8, "250 MB/s", fontsize=8, color="0.25")
ax.axvline(CAN / 1e6, color="0.4", lw=1, ls=":")
ax.text(CAN / 1e6 * 1.08, 8, "1.1 GB/s", fontsize=8, color="0.25")
ax.set_ylim(0, 108)
ax.set_xlabel("băng thông lưu trữ cấp được (MB/s, thang log)")
ax.set_ylabel("mức sử dụng bộ tăng tốc (%)")
ax.set_title("Phương trình 5 và 6: dưới điểm gãy, thêm compute không giúp gì")
ax.legend(loc="upper left", fontsize=8, frameon=False)
plt.tight_layout()
plt.show()
print(f"ở 250 MB/s: {toc_do_huan_luyen(R_ANH, CAP, byte_moi_anh)[0]:,.0f} img/s, "
      f"chặn bởi {toc_do_huan_luyen(R_ANH, CAP, byte_moi_anh)[1]}")

Dưới điểm gãy, đường đi lên tuyến tính theo băng thông và không phụ thuộc gì vào bộ tăng tốc: một
chip nhanh gấp đôi vẫn cho đúng ngần ấy ảnh mỗi giây. Phụ trội (một giả định của sổ tay, để thấy
hình dạng) đẩy điểm gãy sang phải. Cách chữa chỉ có hai hướng, như chương nói: tăng băng thông lưu
trữ dùng được, hoặc giảm số byte mỗi mẫu.

## 4. Ngân sách băng thông lưu trữ cho ResNet-50

Napkin math 1.9 đi theo chiều ngược lại: bắt đầu từ trần compute, rồi đổi nó thành một yêu cầu cho
tầng lưu trữ. Bộ tăng tốc tham chiếu cho 312 TFLOP/s trên phép FP16 dày; một bước huấn luyện
ResNet-50 tốn khoảng 24.6 GFLOP mỗi ảnh, tính cả lượt ngược.

**Dự đoán:** một ổ SATA SSD 500 MB/s giữ bộ tăng tốc đó bận được bao nhiêu phần trăm thời gian?

In [ ]:
R_DINH = sach(312, nguon="312 TFLOP/s · ~8 GFLOP", tol=0.5) * 1e12
FLOP_ANH = sach(24.6, nguon="24,6 GFLOP/ảnh · 12.682 img/s") * 1e9
print(f"lượt xuôi cộng lượt ngược: {FLOP_ANH / FLOP_XUOI:.2f} lần lượt xuôi")
tran = math.floor(R_DINH / FLOP_ANH)          # giới hạn trên, nên làm tròn xuống
theo_sach("trần compute (img/s)", tran, sach=12_682, nguon="24,6 GFLOP/ảnh · 12.682 img/s")
can_19 = tran * ANH_BYTE
theo_sach("băng thông cần (GB/s)", can_19 / GB, sach=1.9, nguon="150 KB · ~1,9 GB/s", tol=0.05)

S3_LUONG = sach(100, trich="100 MB/s per thread") * 1e6
theo_sach("số luồng S3 cần", can_19 / S3_LUONG, sach=19, trich="19 concurrent worker threads")
SATA = sach(500, nguon="500 MB/s SATA III (trần giao diện 550 MB/s)") * 1e6
r_sata = SATA / ANH_BYTE
theo_sach("SATA: ảnh mỗi giây", r_sata, sach=3333, trich="3,333 img/s")
theo_sach("SATA: mức sử dụng bộ tăng tốc (%)", 100 * r_sata / tran, sach=26, trich="26 percent utilization")
CHAM = sach(125, nguon="125 MB/s → 93,4 % thời gian rỗi") * 1e6
theo_sach("nhàn rỗi khi chỉ cấp 125 MB/s (%)", 100 * (1 - CHAM / can_19), sach=93.4,
          nguon="125 MB/s → 93,4 % thời gian rỗi")
NVME = (sach(5, nguon="NVMe 100–300 đô la/TB/tháng · 5–7 GB/s"), sach(7, nguon="NVMe 100–300 đô la/TB/tháng · 5–7 GB/s"))
assert NVME[0] * GB >= can_19   # NVMe đúng là loại lưu trữ cục bộ đủ cho mục tiêu này
print(f"NVMe {NVME[0]}–{NVME[1]} GB/s: dư {NVME[0] * GB / can_19:.1f}–{NVME[1] * GB / can_19:.1f} lần so với nhu cầu")

Đó là lần thứ ba chương nhìn lại cùng một lập luận về bộ tăng tốc bị đói: thuế tiếp liệu ở phần
vật lý, điểm nghẽn giải mã ở phần nạp dữ liệu (sổ tay 02 đo nó trên CPU), và giờ là ngân sách
băng thông. Chương còn một ví dụ nhỏ hơn cùng công thức, và một quy tắc tỉ lệ:

In [ ]:
R_1000 = sach(1000, trich="1,000 images per second at 150 KB each requires 150 MB/s")
theo_sach("1,000 ảnh/s nhân 150 KB (MB/s)", R_1000 * ANH_BYTE / 1e6, sach=150, trich="requires 150 MB/s sustained")
LECH = sach(10, trich="A 10-fold mismatch")
theo_sach("bộ tăng tốc nhanh gấp 10 lần tầng lưu trữ: nhàn rỗi (%)", 100 * (1 - 1 / LECH), sach=90, trich="idle 90 percent")

# vòng đời: huấn luyện trên 8 GPU, rồi phục vụ một hệ thống gợi ý
R_8GPU = sach(40_000, nguon="40.000 img/s trên 8 GPU · 10.000 yêu cầu/giây")
theo_sach("40,000 img/s nhân 150 KB (GB/s)", R_8GPU * ANH_BYTE / GB, sach=6, trich="roughly 6 GB/s", tol=0.5)
YEU_CAU = sach(10_000, nguon="40.000 img/s trên 8 GPU · 10.000 yêu cầu/giây")
DOC_MOI_YC = sach(10, trich="10 feature reads/request")
theo_sach("phép vào ra mỗi giây khi phục vụ", YEU_CAU * DOC_MOI_YC, sach=100_000, nguon="10 ms · 100.000 IOPS")

In [ ]:
cac_B = np.logspace(7.5, 10.2, 300)
muc_su_dung = 100 * np.minimum(1, cac_B / can_19)
fig, ax = plt.subplots(figsize=(8, 4.2))
ax.semilogx(cac_B / 1e6, muc_su_dung, color=MAU[0], lw=2)
ax.axvspan(NVME[0] * 1e3, NVME[1] * 1e3, color=MAU[2], alpha=0.15, lw=0)
ax.text(NVME[0] * 1e3 * 1.03, 45, "NVMe\n5–7 GB/s", fontsize=8, color="0.2")
for B, chu, dy in ((S3_LUONG, "S3, 1 luồng", 62), (CHAM, "125 MB/s", 34), (SATA, "SATA 500 MB/s", 14)):
    u = 100 * min(1, B / can_19)
    ax.plot(B / 1e6, u, "o", color=MAU[1], ms=7, mec="white", mew=1.2)
    ax.annotate(f"{chu}: {u:.0f} %", (B / 1e6, u), xytext=(-10, dy), textcoords="offset points", fontsize=8,
                ha="right", arrowprops=dict(arrowstyle="-", color="0.5", lw=0.7))
ax.axvline(can_19 / 1e6, color="0.4", lw=1, ls="--")
ax.text(can_19 / 1e6 * 0.93, 103, f"cần {can_19 / GB:.1f} GB/s", fontsize=8, color="0.25", ha="right")
ax.set_ylim(0, 112)
ax.set_xlabel("băng thông đọc tuần tự của tầng lưu trữ (MB/s, thang log)")
ax.set_ylabel("mức sử dụng bộ tăng tốc (%)")
ax.set_title("Napkin math 1.9: 12,682 img/s nhân 150 KB mỗi ảnh")
plt.tight_layout()
plt.show()

## 5. Lưu trữ phân tầng: giá và tốc độ của năm tầng

Bảng 8 của chương cho năm tầng lưu trữ: NVMe cục bộ, kho lưu trữ đối tượng (object storage), kho
dữ liệu (data warehouse), cache trong bộ nhớ và lưu trữ lâu dài. Mỗi tầng có một khoảng giá, một
khoảng thông lượng đọc tuần tự và một khoảng độ trễ. Đây là các khoảng để lập kế hoạch năm 2024, đổi
theo nhà cung cấp, vùng và cấu hình, không phải báo giá.

In [ ]:
TANG = pd.DataFrame([
    {"tầng": "NVMe cục bộ", "giá thấp": sach(100, nguon="NVMe 100–300 đô la/TB/tháng"),
     "giá cao": sach(300, nguon="NVMe 100–300 đô la/TB/tháng"),
     "đọc thấp (MB/s)": sach(5, nguon="NVMe 100–300 đô la/TB/tháng · 5–7 GB/s") * 1000,
     "đọc cao (MB/s)": sach(7, nguon="NVMe 100–300 đô la/TB/tháng · 5–7 GB/s") * 1000},
    {"tầng": "object storage", "giá thấp": sach(20, nguon="Object 20–25 đô la · 100–500 MB/s"),
     "giá cao": sach(25, nguon="Object 20–25 đô la · 100–500 MB/s"),
     "đọc thấp (MB/s)": sach(100, nguon="Object 20–25 đô la · 100–500 MB/s"),
     "đọc cao (MB/s)": sach(500, nguon="Object 20–25 đô la · 100–500 MB/s")},
    {"tầng": "kho dữ liệu", "giá thấp": sach(20, nguon="Warehouse 20–40 đô la · 1–5 GB/s"),
     "giá cao": sach(40, nguon="Warehouse 20–40 đô la · 1–5 GB/s"),
     "đọc thấp (MB/s)": sach(1, nguon="Warehouse 20–40 đô la · 1–5 GB/s") * 1000,
     "đọc cao (MB/s)": sach(5, nguon="Warehouse 20–40 đô la · 1–5 GB/s") * 1000},
    {"tầng": "cache trong bộ nhớ", "giá thấp": sach(500, nguon="Cache 500–1.000 đô la · 20–50 GB/s"),
     "giá cao": sach(1000, nguon="Cache 500–1.000 đô la · 20–50 GB/s"),
     "đọc thấp (MB/s)": sach(20, nguon="Cache 500–1.000 đô la · 20–50 GB/s") * 1000,
     "đọc cao (MB/s)": sach(50, nguon="Cache 500–1.000 đô la · 20–50 GB/s") * 1000},
    {"tầng": "lưu trữ lâu dài", "giá thấp": sach(1, nguon="Archive 1–4 đô la · 10–50 MB/s"),
     "giá cao": sach(4, nguon="Archive 1–4 đô la · 10–50 MB/s"),
     "đọc thấp (MB/s)": sach(10, nguon="Archive 1–4 đô la · 10–50 MB/s"),
     "đọc cao (MB/s)": sach(50, nguon="Archive 1–4 đô la · 10–50 MB/s")},
]).set_index("tầng")
print("giá tính bằng $ mỗi TB mỗi tháng")
TANG

Chương áp bảng này vào case phát hiện từ khoá: tập huấn luyện âm thô 748.8 GB. Giá object storage
ở đây là 23 $ mỗi TB mỗi tháng, mức lưu trữ nóng trong bảng 5, nằm trong khoảng 20–25 $ của bảng 8.
Tốc độ object storage điển hình là đầu thấp 100 MB/s của khoảng.

In [ ]:
KWS_GB = sach(748.8, nguon="748,8 GB = 17,2 đô la/tháng object")
GIA_S3 = sach(23, nguon="23 đô la/TB/tháng")
theo_sach("object storage ($/tháng)", KWS_GB / 1000 * GIA_S3, sach=17.2, nguon="748,8 GB = 17,2 đô la/tháng object")
theo_sach("NVMe, đầu thấp ($/tháng)", KWS_GB / 1000 * TANG.loc["NVMe cục bộ", "giá thấp"], sach=74.9,
          nguon="74,9–224,6 đô la/tháng NVMe")
theo_sach("NVMe, đầu cao ($/tháng)", KWS_GB / 1000 * TANG.loc["NVMe cục bộ", "giá cao"], sach=224.6,
          nguon="74,9–224,6 đô la/tháng NVMe")
nap_nvme = KWS_GB * GB / (TANG.loc["NVMe cục bộ", "đọc thấp (MB/s)"] * 1e6)
nap_s3 = KWS_GB * GB / (TANG.loc["object storage", "đọc thấp (MB/s)"] * 1e6)
theo_sach("nạp cả tập ở 5 GB/s (giây)", nap_nvme, sach=149.8, nguon="149,8 giây ở 5 GB/s vs 7.488 giây · 50×")
theo_sach("nạp cả tập ở 100 MB/s (giây)", nap_s3, sach=7488, nguon="149,8 giây ở 5 GB/s vs 7.488 giây · 50×")
theo_sach("chênh lệch", nap_s3 / nap_nvme, sach=50, nguon="149,8 giây ở 5 GB/s vs 7.488 giây · 50×")

Mở rộng ra cả năm tầng: với cùng 748.8 GB, mỗi tầng tốn bao nhiêu mỗi tháng, và một lượt đọc hết
tập mất bao lâu? Mỗi hình chữ nhật dưới đây trải từ đầu rẻ và chậm tới đầu đắt và nhanh của khoảng
trong bảng 8. Với lưu trữ lâu dài, chương ghi việc lấy dữ liệu ra mất hàng giờ, chưa tính ở đây.

In [ ]:
from matplotlib.patches import Rectangle

bang = TANG.assign(**{
    "$/tháng thấp": KWS_GB / 1000 * TANG["giá thấp"], "$/tháng cao": KWS_GB / 1000 * TANG["giá cao"],
    "đọc hết, nhanh nhất": [doc_thoi_gian(KWS_GB * 1000 / v) for v in TANG["đọc cao (MB/s)"]],
    "đọc hết, chậm nhất": [doc_thoi_gian(KWS_GB * 1000 / v) for v in TANG["đọc thấp (MB/s)"]],
    "lượt đọc mỗi ngày, đầu chậm": NGAY / (KWS_GB * 1000 / TANG["đọc thấp (MB/s)"])})
print(bang[["$/tháng thấp", "$/tháng cao", "đọc hết, nhanh nhất", "đọc hết, chậm nhất",
            "lượt đọc mỗi ngày, đầu chậm"]].round(1).to_string())

fig, ax = plt.subplots(figsize=(8, 4.6))
for (ten, h), mau in zip(bang.iterrows(), MAU + ["0.45"]):
    x0, x1 = KWS_GB * 1000 / h["đọc cao (MB/s)"], KWS_GB * 1000 / h["đọc thấp (MB/s)"]
    y0, y1 = h["$/tháng thấp"], h["$/tháng cao"]
    ax.add_patch(Rectangle((x0, y0), x1 - x0, y1 - y0, color=mau, alpha=0.35, lw=0))
    ax.add_patch(Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False, edgecolor=mau, lw=1.5))
    if ten in ("kho dữ liệu", "lưu trữ lâu dài"):   # nhãn bên phải sẽ đè lên hộp kế bên
        ax.text(np.sqrt(x0 * x1), y1 * 1.15, ten, fontsize=8, ha="center", va="bottom", color="0.15")
    else:
        ax.text(x1 * 1.12, np.sqrt(y0 * y1), ten, fontsize=8, va="center", color="0.15")
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlim(5, 2e5)
ax.set_ylim(0.3, 2000)
ax.set_xlabel("thời gian đọc hết 748.8 GB (giây, thang log)")
ax.set_ylabel("$ mỗi tháng để giữ 748.8 GB (thang log)")
ax.set_title("Bảng 8 áp vào tập dữ liệu KWS: không tầng nào vừa rẻ vừa nhanh")
plt.tight_layout()
plt.show()

DIA = (sach(100, nguon="100–200 MB/s đĩa quay vs 50–200 GB/s RAM"), sach(200, nguon="100–200 MB/s đĩa quay vs 50–200 GB/s RAM"))
RAM = (sach(50, nguon="100–200 MB/s đĩa quay vs 50–200 GB/s RAM"), sach(200, nguon="100–200 MB/s đĩa quay vs 50–200 GB/s RAM"))
print(f"bậc thang băng thông: đĩa quay đọc hết 748.8 GB trong {doc_thoi_gian(KWS_GB * 1000 / DIA[1])}"
      f"–{doc_thoi_gian(KWS_GB * 1000 / DIA[0])}, RAM trong {KWS_GB / RAM[1]:.1f}–{KWS_GB / RAM[0]:.1f} giây")

Các tầng xếp gần như trên một đường chéo: muốn đọc nhanh hơn một bậc thì trả đắt hơn khoảng một
bậc. Vì thế hệ thống học máy dùng lưu trữ phân tầng (tiered storage): giữ âm thô rẻ trên object
storage, chép tập đang huấn luyện lên NVMe, và chỉ để trong bộ nhớ những gì phục vụ phải lấy trong
vài micro giây. Ở đầu chậm, object storage cho chưa tới mười lượt đọc hết tập mỗi ngày, còn NVMe cho
hàng trăm: đó là khác biệt giữa lặp nhiều lần mỗi ngày và chờ hàng giờ giữa hai thí nghiệm.

## 6. Bậc thang độ trễ trên thang thời gian của con người

Bảng 9 quy mọi độ trễ về thang người, lấy một lần đọc cache L1 0.5 ns làm một giây. Hệ số đổi là
$1\text{ s} / 0.5\text{ ns} = 2 \times 10^9$.

In [ ]:
L1 = sach(0.5, nguon="L1 0,5 ns → Internet 100.000.000 ns · thang người: 1 giây → 6 năm")
MOT_GIAY = sach(1, nguon="L1 0,5 ns → Internet 100.000.000 ns · thang người: 1 giây → 6 năm")
HE_SO = MOT_GIAY / (L1 * 1e-9)
BAC = [("cache L1", L1),
       ("cache L2", sach(7, trich="L2 Cache Reference | 7")),
       ("DRAM", sach(100, trich="Main Memory (DRAM) | 100")),
       ("ổ NVMe cục bộ", sach(100_000, trich="SSD (local NVMe) | 100,000")),
       ("mạng trong một trung tâm dữ liệu", sach(500_000, trich="Network (same DC) | 500,000")),
       ("ổ SSD qua mạng", sach(2_000_000, trich="SSD (remote network) | 2,000,000")),
       ("object storage", sach(20_000_000, trich="Object Store (S3) | 20,000,000")),
       ("Internet, California tới Virginia",
        sach(100_000_000, nguon="L1 0,5 ns → Internet 100.000.000 ns · thang người: 1 giây → 6 năm"))]
nguoi = {ten: ns * 1e-9 * HE_SO for ten, ns in BAC}

theo_sach("cache L2 trên thang người (giây)", nguoi["cache L2"], sach=14, trich="7 | 14 seconds")
theo_sach("mạng trong trung tâm dữ liệu (ngày)", nguoi["mạng trong một trung tâm dữ liệu"] / NGAY, sach=11.57,
          trich="11.57 days")
theo_sach("SSD qua mạng (ngày)", nguoi["ổ SSD qua mạng"] / NGAY, sach=46.30, trich="46.30 days")
# bốn dòng sau, bảng ghi tròn về một đơn vị nguyên
theo_sach("DRAM (phút)", nguoi["DRAM"] / 60, sach=3, trich="100 | 3 minutes", tol=0.5)
theo_sach("NVMe cục bộ (ngày)", nguoi["ổ NVMe cục bộ"] / NGAY, sach=2, trich="100,000 | 2 days", tol=0.5)
theo_sach("object storage (năm)", nguoi["object storage"] / (365 * NGAY), sach=1, trich="20,000,000 | 1 year", tol=0.5)
theo_sach("Internet (năm)", nguoi["Internet, California tới Virginia"] / (365 * NGAY), sach=6,
          nguon="L1 0,5 ns → Internet 100.000.000 ns · thang người: 1 giây → 6 năm", tol=0.5)
TAM = sach(8, nguon="hơn 8 bậc độ lớn")
bac_do_lon = math.log10(BAC[-1][1] / L1)
assert bac_do_lon > TAM
print(f"từ cache L1 tới Internet: {bac_do_lon:.1f} bậc độ lớn")
print(f"từ cache L1 tới object storage: {math.log10(BAC[-2][1] / L1):.1f} bậc độ lớn")
print(f"chương: từ thanh ghi trên chip tới lưu trữ ở xa, hơn {TAM} bậc (bảng 9 không có dòng thanh ghi)")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.2))
y = np.arange(len(BAC))[::-1]
ax.barh(y, [ns for _, ns in BAC], color=MAU[0], height=0.6, left=0.1)
ax.set_xscale("log")
ax.set_yticks(y, [ten for ten, _ in BAC], fontsize=8)
for yi, (ten, ns) in zip(y, BAC):
    ax.text(ns * 1.6, yi, doc_thoi_gian(nguoi[ten]), va="center", fontsize=8, color="0.2")
ax.set_xlim(0.1, 5e9)
ax.set_xlabel("độ trễ (ns, thang log); nhãn là thời gian tương ứng trên thang người")
ax.set_title("Bảng 9: lấy 0.5 ns làm một giây")
plt.tight_layout()
plt.show()

Trên thang này, một lần đọc NVMe là hơn 2 ngày, một lần đọc object storage là hơn 1 năm. Một vòng
lặp huấn luyện chờ từng yêu cầu vào ra mà không đệm sẽ để bộ tăng tốc nhàn rỗi gần như suốt. Vì thế
nạp trước (prefetching) và gom yêu cầu thành dòng đọc tuần tự là điều kiện để bộ tăng tốc chạy được,
chứ không phải một tối ưu làm thêm.

## 7. Theo lô hay theo luồng: giá của độ mới

Napkin math 1.5: một quy trình nạp 1M sự kiện mỗi giây, mỗi sự kiện 1 KB. Nạp theo luồng (stream
ingestion) cần độ trễ dưới 100 ms; nạp theo lô (batch ingestion) gom mỗi giờ một lần và xử lý cửa
sổ đó trong 10 phút.

**Dự đoán:** với cùng khối lượng dữ liệu, theo luồng đắt hơn theo lô bao nhiêu lần?

In [ ]:
SU_KIEN = sach(1e6, nguon="1M sự kiện/giây · 1 KB/sự kiện · 1 GB/s")
KB_SK = sach(1, nguon="1M sự kiện/giây · 1 KB/sự kiện · 1 GB/s") * 1e3
luu_luong = SU_KIEN * KB_SK
theo_sach("lưu lượng (GB/s)", luu_luong / GB, sach=1, nguon="1M sự kiện/giây · 1 KB/sự kiện · 1 GB/s")

GIA_LOI = sach(0.05, nguon="50 + 50 = 100 lõi · 0,05 đô la/giờ")
LOI_LUONG = sach(50, nguon="50 + 50 = 100 lõi") + sach(50, nguon="50 + 50 = 100 lõi")   # chính + dự phòng
theo_sach("theo luồng ($/ngày)", LOI_LUONG * 24 * GIA_LOI, sach=120, nguon="0,05 đô la/giờ · 120 đô la/ngày")

cua_so = luu_luong * 3600
theo_sach("một cửa sổ một giờ (TB)", cua_so / TB, sach=3.6, nguon="200 lõi · 3,6 TB")
LOI_LO = sach(200, nguon="120 đô la/ngày vs 200 lõi")
PHUT_XU_LY = sach(10, trich="in 10 minutes")
gio_loi_ngay = LOI_LO * PHUT_XU_LY / 60 * 24
theo_sach("theo lô (giờ-lõi mỗi ngày)", gio_loi_ngay, sach=800, nguon="3,6 TB · 800 giờ-lõi")
chi_lo = gio_loi_ngay * GIA_LOI
theo_sach("theo lô ($/ngày)", chi_lo, sach=40, nguon="800 giờ-lõi · 40 đô la/ngày")
theo_sach("theo luồng so với theo lô", LOI_LUONG * 24 * GIA_LOI / chi_lo, sach=3, nguon="40 đô la/ngày · ~3×", tol=0.05)
print(f"mỗi lõi: theo luồng {luu_luong / sach(50, nguon='50 + 50 = 100 lõi') / 1e6:.0f} MB/s (không tính dự phòng), "
      f"theo lô {cua_so / (LOI_LO * PHUT_XU_LY * 60) / 1e6:.0f} MB/s")

Một lõi theo lô đẩy được nhiều byte hơn một lõi theo luồng, vì vào ra tuần tự với thông lượng cao
là việc hiệu quả, và lõi theo lô chỉ chạy 10 phút mỗi giờ. Theo luồng thì phải giữ lõi luôn bật, cả
lõi dự phòng, để không lúc nào trễ quá 100 ms.

Vậy sao không gom lô mỗi phút, vừa mới vừa rẻ? Phép tính của chương chỉ đếm phần xử lý, nên nó tỉ
lệ với số byte chứ không với số lần chạy. Một lượt chạy lô thật còn tốn phần khởi động: lập lịch,
cấp máy, mở file. Sổ tay **giả định** mỗi lượt tốn thêm 2 phút của 200 lõi, rồi quét chu kỳ gom lô.
Độ mới đo bằng tuổi trung bình của dữ liệu khi đặc trưng dùng được: nửa chu kỳ chờ, cộng thời gian
xử lý.

In [ ]:
KHOI_DONG_PHUT = 2.0   # giả định của sổ tay
cac_chu_ky = {"24 giờ": 24.0, "6 giờ": 6.0, "1 giờ": 1.0, "15 phút": 0.25, "5 phút": 5 / 60, "1 phút": 1 / 60}
dong = []
for ten, gio in cac_chu_ky.items():
    lan_moi_ngay = 24 / gio
    phut_xu_ly = PHUT_XU_LY * gio                     # cùng 200 lõi, khối lượng tỉ lệ với chu kỳ
    gio_loi = LOI_LO * lan_moi_ngay * (phut_xu_ly + KHOI_DONG_PHUT) / 60
    tuoi = gio * 3600 / 2 + (phut_xu_ly + KHOI_DONG_PHUT) * 60
    dong.append({"chu kỳ gom lô": ten, "$/ngày": gio_loi * GIA_LOI, "tuổi trung bình của dữ liệu": doc_thoi_gian(tuoi),
                 "tuổi (giây)": tuoi})
bang_lo = pd.DataFrame(dong)
chi_luong = LOI_LUONG * 24 * GIA_LOI
TRE_LUONG = sach(100, trich="less than 100 ms latency") / 1000
print(bang_lo[["chu kỳ gom lô", "$/ngày", "tuổi trung bình của dữ liệu"]].round(1).to_string(index=False))
print(f"theo luồng: {chi_luong:.0f} $/ngày, dữ liệu dùng được sau dưới {TRE_LUONG * 1000:.0f} ms")

fig, ax = plt.subplots(figsize=(8, 4.2))
ax.loglog(bang_lo["tuổi (giây)"], bang_lo["$/ngày"], "o-", color=MAU[0], lw=2, ms=7, label="theo lô, giả định 2 phút khởi động")
for _, h in bang_lo.iterrows():
    ax.annotate(h["chu kỳ gom lô"], (h["tuổi (giây)"], h["$/ngày"]), xytext=(4, 7), textcoords="offset points", fontsize=8)
ax.plot([TRE_LUONG], [chi_luong], "s", color=MAU[1], ms=8, label="theo luồng (napkin math 1.5)")
ax.axhline(chi_luong, color=MAU[1], lw=1, ls=":")
ax.set_xlim(0.04, 4e5)
ax.set_ylim(30, 900)
ax.set_yticks([40, 60, 100, 200, 400, 800], ["40", "60", "100", "200", "400", "800"])
ax.yaxis.set_minor_formatter(plt.NullFormatter())
ax.set_xlabel("tuổi trung bình của dữ liệu khi dùng được (giây, thang log)")
ax.set_ylabel("$ mỗi ngày (thang log)")
ax.set_title("Mua độ mới bằng tiền: mỗi bậc chu kỳ ngắn hơn đắt thêm")
ax.legend(loc="upper right", fontsize=8, frameon=False)
plt.tight_layout()
plt.show()

Với giả định này, gom lô dày tới vài phút một lần đã đắt ngang hoặc hơn theo luồng mà dữ liệu vẫn
cũ hàng phút. Con số chính xác tuỳ phần khởi động thật của hệ bạn, nhưng hình dạng thì giữ: độ mới
có giá, và chương nói khoản thuế đó chỉ chính đáng khi giá trị của độ trễ dưới một giây vượt phần chi
phí tăng thêm.

## 8. ETL hay ELT: chuyển đổi ở đâu thì rẻ

Napkin math 1.6: mỗi ngày 10 TB dữ liệu luồng nhấp chuột, 3 mô hình, mỗi mô hình một cửa sổ tổng hợp
khác nhau. ETL (trích xuất, chuyển đổi, tải) tính cả ba cửa sổ trong một cụm Spark rồi mới nạp vào
kho. ELT (trích xuất, tải, chuyển đổi) nạp dữ liệu thô trước rồi chuyển đổi trong kho bằng truy vấn.

In [ ]:
TB_NGAY = sach(10, nguon="10 TB/ngày · 3 mô hình")
SO_MO_HINH = sach(3, nguon="10 TB/ngày · 3 mô hình")
GIA_SPARK = sach(5, nguon="5 đô la/TB compute Spark")
NGAY_THANG = sach(30, trich="30 days retention")
spark_ngay = TB_NGAY * GIA_SPARK
theo_sach("ETL: Spark ($/ngày)", spark_ngay, sach=50, nguon="3 mô hình · 50 đô la/ngày")
theo_sach("ETL: Spark ($/tháng)", spark_ngay * NGAY_THANG, sach=1500, nguon="1.500 đô la/tháng compute")
TB_MOI_TAP = sach(2, nguon="3 × ~2 TB = 6 TB", tol=0.5)
theo_sach("ETL: dung lượng đã chuyển đổi (TB)", SO_MO_HINH * TB_MOI_TAP, sach=6, nguon="3 × ~2 TB = 6 TB", tol=0.5)
etl_luu = SO_MO_HINH * TB_MOI_TAP * GIA_S3
theo_sach("ETL: lưu trữ ($/tháng)", etl_luu, sach=138, nguon="50 đô la/ngày · 138 đô la/tháng")

elt_tb = TB_NGAY * NGAY_THANG
theo_sach("ELT: dữ liệu thô giữ 30 ngày (TB)", elt_tb, sach=300, nguon="138 đô la/tháng · 300 TB")
elt_luu = elt_tb * GIA_S3
theo_sach("ELT: lưu trữ ($/tháng)", elt_luu, sach=6900, nguon="300 TB · 6.900 đô la/tháng")
GIA_TRUY_VAN = sach(5, trich="$5/query")
elt_truy_van = SO_MO_HINH * GIA_TRUY_VAN * NGAY_THANG
theo_sach("ELT: compute truy vấn ($/tháng)", elt_truy_van, sach=450, nguon="6.900 đô la/tháng · 450 đô la/tháng")

etl = spark_ngay * NGAY_THANG + etl_luu
elt = elt_luu + elt_truy_van
theo_sach("ETL tổng ($/tháng)", etl, sach=1638, nguon="1.638 vs 7.350 đô la/tháng")
theo_sach("ELT tổng ($/tháng)", elt, sach=7350, nguon="1.638 vs 7.350 đô la/tháng")
theo_sach("lợi thế của ETL ($/tháng)", elt - etl, sach=5712, nguon="lợi 5.712 đô la/tháng")
theo_sach("ETL tiết kiệm riêng phần lưu trữ ($/tháng)", elt_luu - etl_luu, sach=6762, nguon="6.762 đô la/tháng lưu trữ")
GIO_ETL = sach(4, trich="Re-run full pipeline (4 hours)")
PHUT_ELT = sach(30, trich="Rewrite SQL query (30 minutes)")
theo_sach("thời gian kỹ sư mỗi lần đổi giản đồ, ETL so với ELT", GIO_ETL * 60 / PHUT_ELT, sach=8,
          nguon="8× thời gian kỹ sư")

Tính trên chi phí cloud, ETL rẻ hơn hẳn. Nhưng mỗi lần đổi giản đồ, ETL tốn thời gian kỹ sư gấp 8
lần, và chương nói điểm hoà vốn thật phụ thuộc vào giá công sức kỹ sư và tần suất đổi giản đồ. Hãy
đặt hai đại lượng đó vào. Sổ tay **giả định** một giờ kỹ sư giá 100 $, rồi thử thêm 50 $ và 200 $.

**Dự đoán:** mỗi tháng phải đổi giản đồ bao nhiêu lần thì ELT bắt đầu rẻ hơn?

In [ ]:
cac_lan = np.arange(0, 41)
fig, ax = plt.subplots(figsize=(8, 4.2))
for gia_gio, mau in zip((50, 100, 200), MAU):          # giả định: $ mỗi giờ kỹ sư
    hoa_von = (elt - etl) / ((GIO_ETL - PHUT_ELT / 60) * gia_gio)
    print(f"giờ kỹ sư {gia_gio:>3} $: ELT rẻ hơn khi đổi giản đồ quá {hoa_von:.1f} lần mỗi tháng")
    ax.plot(cac_lan, etl + cac_lan * GIO_ETL * gia_gio, color=mau, lw=2, label=f"ETL, giờ kỹ sư {gia_gio} $")
    ax.plot(cac_lan, elt + cac_lan * PHUT_ELT / 60 * gia_gio, color=mau, lw=2, ls="--")
    ax.plot(hoa_von, etl + hoa_von * GIO_ETL * gia_gio, "o", color=mau, ms=7, mec="white", mew=1.2)
ax.plot([], [], color="0.4", ls="--", label="ELT (cùng màu, nét đứt)")
ax.set_xlabel("số lần đổi giản đồ mỗi tháng")
ax.set_ylabel("tổng chi phí mỗi tháng ($)")
ax.set_title("ETL rẻ khi giản đồ ổn định, ELT rẻ khi định nghĩa đặc trưng đổi thường xuyên")
ax.legend(loc="upper left", fontsize=8, frameon=False)
plt.tight_layout()
plt.show()

Mỗi chấm là một điểm hoà vốn. Giá công sức càng cao, điểm hoà vốn càng tới sớm, đúng kết luận của
chương: giản đồ ổn định thì hợp với chi phí cloud thấp của ETL, còn định nghĩa đặc trưng đổi thường
xuyên thì hợp với tốc độ lặp của ELT. Phép tính này vẫn chưa tính một lợi thế của ELT mà chương nêu:
dữ liệu thô còn đó, nên một đặc trưng mới tính được trên lịch sử mà không phải nạp lại từ nguồn.

## Thử thêm

1. Ở phần 4, đổi `ANH_BYTE` thành `50e3` (ảnh nén mạnh hơn, một giả định). SATA SSD còn là điểm
   nghẽn không? Bộ tăng tốc phải trả gì cho việc giải nén mạnh hơn? Sổ tay 02 đo phần đó.
2. Ở phần 7, đổi `KHOI_DONG_PHUT` thành `0.5` rồi `10`. Chu kỳ gom lô nào bắt đầu đắt hơn theo luồng?
3. Ở phần 1, thêm đường 400 Gb/s vào `duong`. Với 1 PB, nó có đưa thời gian truyền xuống dưới thời
   gian huấn luyện điển hình của bạn không?

## Tóm lại

* Trọng lực dữ liệu là một phép chia: 1 PB qua 100 Gb/s mất khoảng 22 giờ và 20,000 $ phí truyền ra,
  nên ở quy mô petabyte, compute chuyển tới chỗ dữ liệu.
* Di chuyển một giá trị 32 bit đắt hơn một phép nhân từ 173 tới 10,810.8 lần; nó chi phối năng lượng
  khi mô hình làm ít phép tính trên mỗi byte, không phải trong mọi trường hợp.
* Thông lượng huấn luyện bị chặn bởi số nhỏ hơn giữa compute và tốc độ cấp dữ liệu: 12,682 img/s nhân
  150 KB cần khoảng 1.9 GB/s, nên SATA SSD giữ bộ tăng tốc bận khoảng 26 % còn NVMe thì đủ.
* Không tầng lưu trữ nào vừa rẻ vừa nhanh, nên dữ liệu được đặt theo tầng: tập KWS 748.8 GB tốn
  17.2 $ mỗi tháng trên object storage, nhưng nạp từ NVMe nhanh gấp 50 lần.
* Theo luồng đắt gấp khoảng 3 lần theo lô cho cùng khối lượng, và ETL rẻ hơn ELT 5,712 $ mỗi tháng
  trên cloud; cả hai kết luận đổi chiều khi đưa độ mới và công sức kỹ sư vào phép tính.

---
### Nguồn và giấy phép

Sổ tay này đi kèm bài học [Kỹ thuật dữ liệu](https://d3lu8vk4we0dc.cloudfront.net/lessons/vol1-ch04-data-engineering/), dựng từ chương
[*Data Engineering*](https://mlsysbook.ai/vol1/data_engineering/data_engineering.html) của sách.

Nội dung gốc thuộc Machine Learning Systems của Vijay Janapa Reddi và cộng sự, MIT Press. Bản quyền © 2024-2026 Harvard University. Giấy phép CC BY-NC-SA 4.0. Bản tiếng Việt này là tác phẩm phái sinh dùng cùng giấy phép: ghi công, chia sẻ tương tự, phi thương mại.